In [1]:
!pip install kagglehub librosa pandas numpy tensorflow scikit-learn tqdm

In [2]:
# ==========================================
# STEP 1: INSTALL & IMPORT LIBRARIES
# ==========================================
import os
import glob
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import kagglehub

# ดาวน์โหลด Dataset จาก Kaggle ลงเครื่อง Colab
path = kagglehub.dataset_download("soumendraprasad/sound-of-114-species-of-birds-till-2022")
print("\n[INFO] Path to dataset files:", path)

# กำหนดตำแหน่งไฟล์ CSV เมทาดาตา
csv_path = os.path.join(path, "Birds Voice.csv")
df = pd.read_csv(csv_path)

100%|██████████| 2.06G/2.06G [01:47<00:00, 20.6MB/s]

Extracting files...



[INFO] Path to dataset files: /root/.cache/kagglehub/datasets/soumendraprasad/sound-of-114-species-of-birds-till-2022/versions/1


In [3]:
# ==========================================
# STEP 2: DATA CLEANING & SUBSET SELECTION
# ==========================================
# กวาดรายชื่อไฟล์เสียงจริงทั้งหมดที่มีอยู่ในโฟลเดอร์ย่อยมาเก็บไว้ก่อน (ป้องกัน Data Leakage)
print("[INFO] กำลังสแกนไฟล์เสียงจริงในระบบ...")
all_audio_files = glob.glob(os.path.join(path, '**', '*.mp3'), recursive=True) + \
                  glob.glob(os.path.join(path, '**', '*.wav'), recursive=True)
print(f"[INFO] เจอไฟล์เสียงจริงในโฟลเดอร์ทั้งหมด: {len(all_audio_files)} ไฟล์")

# เลือกนก 5 สายพันธุ์ที่มีจำนวนข้อมูลเยอะที่สุดมาเล่น
top_5_birds = df['common_name'].value_counts().head(5).index.tolist()
bird_to_label = {name: idx for idx, name in enumerate(top_5_birds)}
print(f"[INFO] 5 สายพันธุ์ยอดนิยมที่เลือกมาเทรน: {top_5_birds}")

[INFO] กำลังสแกนไฟล์เสียงจริงในระบบ...
[INFO] เจอไฟล์เสียงจริงในโฟลเดอร์ทั้งหมด: 2161 ไฟล์
[INFO] 5 สายพันธุ์ยอดนิยมที่เลือกมาเทรน: ['Great Tinamou', 'Solitary Tinamou', 'White-throated Tinamou', 'Grey Tinamou', 'Small-billed Tinamou']


In [4]:
# ==========================================
# STEP 3: FEATURE EXTRACTION (MFCC)
# ==========================================
def extract_mfcc(file_path, max_pad_len=200):
    """ฟังก์ชันโหลดไฟล์เสียงและสกัดฟีเจอร์ MFCC ความยาว 5 วินาที"""
    try:
        # โหลดเสียงจำกัดที่ 5 วินาที และใช้ Sampling Rate 16kHz ตามมาตรฐาน TinyML
        audio, sample_rate = librosa.load(file_path, sr=16000, duration=5.0)

        # สกัดค่า MFCC จำนวน 13 Coefficients เพื่อให้โมเดลมีขนาดเล็กประมวลผลบนบอร์ดได้ง่าย
        mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=13)

        # ทำ Padding หรือ Truncating ให้จำนวนเฟรมเวลาเท่ากันทุกไฟล์
        if mfcc.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_pad_len]

        return mfcc.T # Transpose ให้ได้มิติ (Time Steps, Features)
    except Exception as e:
        return None

X = []
y = []

print("\n[INFO] กำลังสกัดฟีเจอร์ MFCC จากไฟล์เสียงจริง (ไม่ซ้ำกัน)...")
for file_path in tqdm(all_audio_files):
    basename = os.path.basename(file_path)

    # ตรวจสอบว่าไฟล์เสียงปัจจุบันตรงกับนก 1 ใน 5 สายพันธุ์ที่เราคัดเลือกไว้หรือไม่
    matched_bird = None
    for bird in top_5_birds:
        if bird in basename:
            matched_bird = bird
            break

    if matched_bird is not None:
        features = extract_mfcc(file_path)
        if features is not None:
            X.append(features)
            y.append(bird_to_label[matched_bird])

X = np.array(X)
y = np.array(y)

print("\n=== สรุปผลการจัดเตรียมชุดข้อมูล ===")
print(f"จำนวนไฟล์ข้อมูลจริงที่ไม่ซ้ำกัน (Samples): {len(X)}")
print(f"รูปร่างของมิติข้อมูล X Input: {X.shape} -> (Samples, Time Steps, Features)")

# ตรวจสอบความพร้อมของข้อมูลก่อนไปต่อ
if len(X) == 0:
    raise ValueError("ไม่พบข้อมูลไฟล์เสียงที่จับคู่ได้ กรุณาตรวจสอบการดาวน์โหลดหรือชื่อไฟล์เสียง")


[INFO] กำลังสกัดฟีเจอร์ MFCC จากไฟล์เสียงจริง (ไม่ซ้ำกัน)...


100%|██████████| 2161/2161 [00:26<00:00, 82.46it/s]  


=== สรุปผลการจัดเตรียมชุดข้อมูล ===
จำนวนไฟล์ข้อมูลจริงที่ไม่ซ้ำกัน (Samples): 150
รูปร่างของมิติข้อมูล X Input: (150, 200, 13) -> (Samples, Time Steps, Features)


In [5]:
# ==========================================
# STEP 4: MODEL TRAINING (1D-CNN) WITH TUNING
# ==========================================
# แบ่งข้อมูลสำหรับ Train 80% และ Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# สถาปัตยกรรม 1D-CNN ที่ปรับแต่งใหม่ (Architecture Tuning)
model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(32, kernel_size=10, activation='relu', input_shape=(X.shape[1], X.shape[2])),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(top_5_birds), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("\n=== โครงสร้างสถาปัตยกรรมแบบจำลองที่ปรับแต่งแล้ว ===")
model.summary()

# -------------------------------------------------------------
# ตั้งค่า Early Stopping เพื่อดักจับและล็อกโมเดลเวอร์ชันที่ฉลาดที่สุด
# -------------------------------------------------------------
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',          # เฝ้าติดตามค่า Loss ของข้อมูล Validation
    patience=5,                  # ถ้า val_loss ไม่ดีขึ้นติดต่อกัน 5 Epoch จะสั่งหยุดเทรนทันที
    restore_best_weights=True,   # สำคัญมาก! บังคับให้โมเดลโหลดน้ำหนักใน Epoch ที่ val_loss ต่ำที่สุดกลับมาหลังหยุดเทรน
    verbose=1
)

print("\n[INFO] เริ่มต้นฝึกสอนแบบจำลองบน GPU T4 (Max 100 Epochs)...")
# ใส่ callbacks=[early_stopping] เข้าไปในกระบวนการ fit
history = model.fit(X_train, y_train,
                    epochs=100,
                    batch_size=16,
                    validation_data=(X_test, y_test),
                    callbacks=[early_stopping])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== โครงสร้างสถาปัตยกรรมแบบจำลองที่ปรับแต่งแล้ว ===


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 191, 32)        │         4,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 95, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 95, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 93, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 46, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 46, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,885 (58.14 KB)

 Trainable params: 14,885 (58.14 KB)

 Non-trainable params: 0 (0.00 B)


[INFO] เริ่มต้นฝึกสอนแบบจำลองบน GPU T4 (Max 100 Epochs)...
Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 509ms/step - accuracy: 0.1500 - loss: 15.8777 - val_accuracy: 0.3000 - val_loss: 4.5336
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3250 - loss: 3.8466 - val_accuracy: 0.2000 - val_loss: 3.3044
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4417 - loss: 2.1996 - val_accuracy: 0.5000 - val_loss: 2.2708
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5583 - loss: 1.5469 - val_accuracy: 0.5333 - val_loss: 2.0068
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5917 - loss: 1.2237 - val_accuracy: 0.5333 - val_loss: 1.7016
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6833 - loss: 0.9752 - val_accuracy: 0.5000 - val_loss: 1.5681
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6333 - loss: 0.9585 - val_accuracy: 0.4667 - val_loss: 1.4477
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accu

In [6]:
# ==========================================
# STEP 5: EVALUATION & EXPORT MODEL
# ==========================================
print("\n=== การประเมินผลประสิทธิภาพแบบจำลอง (Evaluation) ===")
y_pred = np.argmax(model.predict(X_test), axis=1)

# 1. แสดงค่าสถิติหลักในการวัดผล (Precision, Recall, F1-Score, Accuracy)
print("\n[1] Classification Report:")
print(classification_report(y_test, y_pred, target_names=top_5_birds))

# 2. บันทึกโมเดลเป็น Native Keras Format (แทนรูปแบบไฟล์ .h5 แบบเก่า)
model_name = "bird_edge_1d_cnn.keras"
model.save(model_name)

# 3. ตรวจสอบขนาดไฟล์ (Memory Footprint) เพื่อประเมินความเป็นไปได้ในการลงบอร์ดฝังตัว
file_size_kb = os.path.getsize(model_name) / 1024
print(f"[2] ขนาดไฟล์โมเดลที่บันทึกสำเร็จ: {file_size_kb:.2f} KB")


=== การประเมินผลประสิทธิภาพแบบจำลอง (Evaluation) ===
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 534ms/step

[1] Classification Report:
                        precision    recall  f1-score   support

         Great Tinamou       0.71      0.83      0.77         6
      Solitary Tinamou       0.86      0.86      0.86         7
White-throated Tinamou       0.62      0.71      0.67         7
          Grey Tinamou       1.00      0.33      0.50         6
  Small-billed Tinamou       0.67      1.00      0.80         4

              accuracy                           0.73        30
             macro avg       0.77      0.75      0.72        30
          weighted avg       0.78      0.73      0.72        30

[2] ขนาดไฟล์โมเดลที่บันทึกสำเร็จ: 215.51 KB


In [7]:
# ==========================================
# STEP 4 & 5: KERNEL TUNING BENCHMARK & EXPORT
# ==========================================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# แบ่งข้อมูลสำหรับ Train 80% และ Test 20% (ยึดตามโครงสร้างเดิม)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# กำหนดค่า kernel_size ที่ต้องการทดสอบเปรียบเทียบ
kernel_sizes_to_test = [5, 10, 15, 20, 25]

# ตัวแปรสำหรับเก็บสถิติผลลัพธ์ของแต่ละ Kernel
benchmark_results = []
best_accuracy = -1.0
best_model = None
best_kernel_size = None

print(f"[INFO] เริ่มต้นระบบทดสอบเปรียบเทียบประสิทธิภาพชุดตัวกรอง (Total: {len(kernel_sizes_to_test)} รอบ)")

for k_size in kernel_sizes_to_test:
    print(f"\n--------------------------------------------------")
    print(f"กำลังทดสอบโมเดลที่ใช้ kernel_size = {k_size}")
    print(f"--------------------------------------------------")

    # สร้างโครงสร้างโมเดล 1D-CNN ตามค่า kernel_size ในรอบนั้นๆ
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(32, kernel_size=k_size, activation='relu', input_shape=(X.shape[1], X.shape[2])),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(len(top_5_birds), activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # ดักจับ Early Stopping ป้องกัน Overfitting เพื่อให้การเปรียบเทียบเป็นธรรมที่สุด
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=0
    )

    # ฝึกสอนโมเดลบน GPU T4
    model.fit(X_train, y_train,
              epochs=100,
              batch_size=16,
              validation_data=(X_test, y_test),
              callbacks=[early_stopping],
              verbose=0) # ปิดการพ่น log ยาวๆ ระหว่างเทรนเพื่อความสะอาดตา

    # ประเมินผลลัพธ์บน Test Set
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    # สกัดค่าสถิติวัดผลหลัก
    acc = report['accuracy']
    macro_f1 = report['macro avg']['f1-score']

    # บันทึกไฟล์ชั่วคราวเพื่อวัดขนาดโมเดลจริง (Memory Footprint)
    temp_filename = f"temp_k_{k_size}.keras"
    model.save(temp_filename)
    file_size_kb = os.path.getsize(temp_filename) / 1024
    os.remove(temp_filename) # ลบไฟล์ชั่วคราวออก

    print(f"-> ผลลัพธ์: Accuracy = {acc:.4f} | Macro F1 = {macro_f1:.4f} | Size = {file_size_kb:.2f} KB")

    # เก็บข้อมูลลงตารางเปรียบเทียบ
    benchmark_results.append({
        'Kernel Size': k_size,
        'Accuracy': acc,
        'Macro F1-Score': macro_f1,
        'Model Size (KB)': file_size_kb
    })

    # ตรวจสอบหาโมเดลที่ให้ค่า Accuracy สูงที่สุดเพื่อเลือกไปบันทึกไฟล์จริง
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_kernel_size = k_size

[INFO] เริ่มต้นระบบทดสอบเปรียบเทียบประสิทธิภาพชุดตัวกรอง (Total: 5 รอบ)

--------------------------------------------------
กำลังทดสอบโมเดลที่ใช้ kernel_size = 5
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


-> ผลลัพธ์: Accuracy = 0.5000 | Macro F1 = 0.4505 | Size = 191.15 KB

--------------------------------------------------
กำลังทดสอบโมเดลที่ใช้ kernel_size = 10
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


-> ผลลัพธ์: Accuracy = 0.6667 | Macro F1 = 0.6686 | Size = 215.53 KB

--------------------------------------------------
กำลังทดสอบโมเดลที่ใช้ kernel_size = 15
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


-> ผลลัพธ์: Accuracy = 0.6667 | Macro F1 = 0.6356 | Size = 239.90 KB

--------------------------------------------------
กำลังทดสอบโมเดลที่ใช้ kernel_size = 20
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


-> ผลลัพธ์: Accuracy = 0.7000 | Macro F1 = 0.7086 | Size = 264.28 KB

--------------------------------------------------
กำลังทดสอบโมเดลที่ใช้ kernel_size = 25
--------------------------------------------------


-> ผลลัพธ์: Accuracy = 0.7667 | Macro F1 = 0.7538 | Size = 288.66 KB


In [8]:
# ==========================================
# STEP 6: SUMMARY REPORT & AUTO-SAVE BEST MODEL
# ==========================================
print("\n==================================================")
print("ตารางสรุปผลการเปรียบเทียบประสิทธิภาพ (BENCHMARK REPORT)")
print("==================================================")
df_results = pd.DataFrame(benchmark_results)
print(df_results.to_string(index=False))

print(f"\n สรุป: โมเดลที่ให้ค่า Accuracy สูงที่สุดคือ kernel_size = {best_kernel_size} (Accuracy: {best_accuracy:.4f})")

# ทำการเซฟโมเดลตัวที่ฉลาดที่สุดลงระบบโดยอัตโนมัติ ตามคำสั่งฟอร์แมตหลักที่คุณต้องการ
final_model_name = "bird_edge_1d_cnn.keras"
best_model.save(final_model_name)


ตารางสรุปผลการเปรียบเทียบประสิทธิภาพ (BENCHMARK REPORT)
 Kernel Size  Accuracy  Macro F1-Score  Model Size (KB)
           5  0.500000        0.450528       191.151367
          10  0.666667        0.668590       215.527344
          15  0.666667        0.635596       239.902344
          20  0.700000        0.708556       264.277344
          25  0.766667        0.753846       288.660156

 สรุป: โมเดลที่ให้ค่า Accuracy สูงที่สุดคือ kernel_size = 25 (Accuracy: 0.7667)


In [9]:
# ==========================================
# STEP 7: EMBEDDED DEPLOYMENT ON ESP32 (TINYML INFERENCE)
# ==========================================
import tensorflow as tf

# 1. โหลดโมเดลตัวที่ดีที่สุดของคุณ
best_model = tf.keras.models.load_model("bird_edge_1d_cnn.keras")

# 2. ตั้งค่า Converter
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

# ❌ ปิดการทำ Quantization เพื่อไม่ให้เกิด Hybrid Model
# converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. ทำการแปลงเป็น TFLite แบบ Float32 เต็มรูปแบบ
tflite_model = converter.convert()

# 4. บันทึกผลลัพธ์
with open("bird_edge_1d_cnn.tflite", "wb") as f:
    f.write(tflite_model)

print("[SUCCESS] แปลงโมเดลเป็น TFLite แบบ Float32 สำเร็จ!")

Saved artifact at '/tmp/tmpdrkomc31'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 13), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  137777264966736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264962896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264955216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264956560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264963664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264959440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264964048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137777264954448: TensorSpec(shape=(), dtype=tf.resource, name=None)
[SUCCESS] แปลงโมเดลเป็น TFLite แบบ Float32 สำเร็จ!


In [10]:
!xxd -i /content/bird_edge_1d_cnn.tflite > bird_edge_1d_cnn.h

In [14]:
# ==========================================
# STEP 8: EXPORT REAL MFCC FEATURES TO C++ ARRAYS FOR ESP32
# ==========================================
import os
import random
import librosa
import numpy as np
from tqdm import tqdm

# 1. ตั้งค่าพารามิเตอร์และการเลือกกลุ่มตัวอย่าง (รวม 10 ไฟล์)
target_birds = ['Great Tinamou', 'Solitary Tinamou', 'White-throated Tinamou', 'Grey Tinamou', 'Small-billed Tinamou']
num_samples_per_target = 1
num_samples_other = 5

selected_files_info = []

# แยกไฟล์ตามเงื่อนไขสายพันธุ์เป้าหมาย
for bird in target_birds:
    bird_files = [f for f in all_audio_files if bird in os.path.basename(f)]
    sampled = random.sample(bird_files, min(len(bird_files), num_samples_per_target))
    for f in sampled:
        selected_files_info.append({'path': f, 'class': bird})

# ดึงไฟล์สายพันธุ์อื่นๆ ที่ไม่มีชื่อนก 5 ชนิดนี้อยู่เลย
other_files = [f for f in all_audio_files if not any(bird in os.path.basename(f) for bird in target_birds)]
sampled_other = random.sample(other_files, min(len(other_files), num_samples_other))
for f in sampled_other:
    selected_files_info.append({'path': f, 'class': 'Other_Species'})

# 2. กระบวนการโหลดเสียง สกัด MFCC และสร้างไฟล์ C++ Header
output_header_path = "esp32_mfcc_samples.h"

print(f"[INFO] กำลังสกัดฟีเจอร์ MFCC จาก {len(selected_files_info)} ไฟล์ ลงไฟล์ '{output_header_path}'...")

with open(output_header_path, "w") as h_file:
    h_file.write("#ifndef ESP32_MFCC_SAMPLES_H\n")
    h_file.write("#define ESP32_MFCC_SAMPLES_H\n\n")
    h_file.write(f"// ข้อมูล MFCC ของจริงจำนวน {len(selected_files_info)} ไฟล์ สำหรับทดสอบ TinyML\n")
    h_file.write(f"#define TOTAL_AUDIO_SAMPLES {len(selected_files_info)}\n\n")

    for idx, sample in enumerate(tqdm(selected_files_info)):
        try:
            # โหลดเสียงความยาว 1.5 วินาที
            audio, sr = librosa.load(sample['path'], sr=16000, duration=1.5)

            # เติมแพดดิ้งให้ครบ 24000 samples กรณีไฟล์สั้นกว่า 1.5 วินาที
            if len(audio) < 24000:
                audio = np.pad(audio, (0, 24000 - len(audio)))

            # สกัด MFCC ให้ได้ 200 เฟรมเวลา และ 13 ฟีเจอร์
            mfccs = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=13, hop_length=120, n_fft=512)

            # สลับแกน (Transpose) และตัดให้เหลือ 200 แถวเป๊ะๆ -> มิติ (200, 13)
            mfccs = mfccs.T[:200, :]

            # ยืดอาเรย์ให้แบนราบเป็น 1D ขนาด 2600 ตัวแปร
            flat_mfcc = mfccs.flatten()

            filename = os.path.basename(sample['path'])
            h_file.write(f"// Sample {idx + 1} | Class: {sample['class']} | File: {filename}\n")
            h_file.write(f"const int audio_len_{idx} = 2600;\n")
            h_file.write(f"const float audio_data_{idx}[] PROGMEM = {{\n    ")

            # ปัดเศษทศนิยมเหลือ 5 ตำแหน่ง ลดขนาดไฟล์ C++
            hex_data = [f"{val:.5f}f" for val in flat_mfcc]

            for i in range(0, len(hex_data), 10):
                line = ", ".join(hex_data[i:i+10])
                if i + 10 < len(hex_data):
                    h_file.write(line + ",\n    ")
                else:
                    h_file.write(line + "\n")

            h_file.write("};\n\n")

        except Exception as e:
            print(f"\n[WARNING] ไม่สามารถประมวลผลไฟล์ {sample['path']} ได้: {e}")

    h_file.write("// ตารางดัชนีชี้ตำแหน่งพอยน์เตอร์ข้อมูล\n")
    h_file.write("const float* const ALL_AUDIO_DATA[] = {\n")
    for i in range(len(selected_files_info)):
        h_file.write(f"    audio_data_{i},\n")
    h_file.write("};\n\n")

    h_file.write("const int ALL_AUDIO_LEN[] = {\n")
    for i in range(len(selected_files_info)):
        h_file.write(f"    audio_len_{i},\n")
    h_file.write("};\n\n")

    h_file.write("#endif // ESP32_MFCC_SAMPLES_H\n")

print(f"\n[SUCCESS] สร้างไฟล์ '{output_header_path}' เรียบร้อยแล้ว")

[INFO] กำลังสกัดฟีเจอร์ MFCC จาก 10 ไฟล์ ลงไฟล์ 'esp32_mfcc_samples.h'...


100%|██████████| 10/10 [00:00<00:00, 27.67it/s]


[SUCCESS] สร้างไฟล์ 'esp32_mfcc_samples.h' เรียบร้อยแล้ว
